In [1]:
pip install -U langgraph langchain langchain-openai tavily-python python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import operator
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from tavily import TavilyClient


In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3
)

tavily = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

In [ ]:
class Task(BaseModel):
    id: int

    title: str

    goal: str = Field(
        description=(
            "What the reader should learn or accomplish "
            "after reading this section."
        )
    )

    bullets: List[str] = Field(
        description="Important points that must be covered."
    )

    image_query: str = Field(
        description=(
            "A specific search query for a useful image, "
            "diagram, screenshot, or illustration for this section."
        )
    )

In [ ]:

class Plan(BaseModel):

    blog_title: str

    audience: str

    tone: str

    tasks: List[Task]


In [ ]:
class State(TypedDict):

    topic: str

    plan: Plan

    research: str

    sections: Annotated[list, operator.add]

    images: Annotated[list, operator.add]

    final: str


In [ ]:
def orchestrator(state: State) -> dict:

    planner = llm.with_structured_output(Plan)

    plan = planner.invoke(
        [
            SystemMessage(
                content="""
You are an expert technical content strategist.

Your job is to create a practical plan for a high-quality
technical blog.

Analyze the topic and determine:

1. What the reader wants to learn
2. The target audience
3. The appropriate tone
4. The concepts that must be covered
5. The logical order of sections
6. Practical examples and real-world use cases
7. Important mistakes, limitations, or misconceptions

Create multiple focused tasks.

Each task must contain:

- A clear section title
- A specific learning goal
- Important bullet points
- A useful image search query

The image query should describe a visual that would genuinely
help the reader understand the section.

Possible visuals include:

- Architecture diagrams
- Technical diagrams
- Screenshots
- Concept illustrations
- Workflow diagrams
- Charts
- Relevant photographs

Do not create generic or decorative image queries.

The plan must be detailed enough for separate worker agents
to independently write each section.

Return ONLY the structured Plan.
"""
            ),
            HumanMessage(
                content=f"""
Create a blog plan for:

Topic:
{state["topic"]}
"""
            )
        ]
    )

    return {
        "plan": plan
    }

In [ ]:
def researcher(state: State) -> dict:

    topic = state["topic"]
    plan = state["plan"]

    search_queries = [
        topic,
        f"{topic} technical explanation",
        f"{topic} best practices",
        f"{topic} real world examples"
    ]

    research_results = []

    for query in search_queries:

        results = tavily.search(
            query=query,
            search_depth="advanced",
            max_results=5
        )

        for result in results["results"]:

            research_results.append(
                {
                    "title": result.get("title"),
                    "url": result.get("url"),
                    "content": result.get("content")
                }
            )

    research_text = "\n\n".join(
        f"""
SOURCE:
{item["title"]}

URL:
{item["url"]}

CONTENT:
{item["content"]}
"""
        for item in research_results
    )

    return {
        "research": research_text
    }


In [ ]:
def fanout(state: State):

    return [
        Send(
            "worker",
            {
                "task": task,
                "topic": state["topic"],
                "plan": state["plan"],
                "research": state["research"]
            }
        )
        for task in state["plan"].tasks
    ]


In [ ]:
def worker(payload: dict) -> dict:

    task = payload["task"]

    topic = payload["topic"]

    plan = payload["plan"]

    research = payload["research"]

    bullets = "\n- " + "\n- ".join(task.bullets)

    result = llm.invoke(
        [
            SystemMessage(
                content="""
You are a professional technical blog writer.

You are one worker in a multi-agent writing system.

Write ONLY the assigned section.

Do not write the entire blog.

Follow the task goal and bullet points.

Requirements:

- Use Markdown
- Start with the section heading
- Explain concepts clearly
- Be technically accurate
- Use the provided research
- Do not invent facts
- Include practical examples when useful
- Include code examples when appropriate
- Avoid unnecessary filler
- Avoid repeating other sections
- Keep the section focused

The section should be useful enough to publish
with minimal editing.

Return ONLY the Markdown content.
"""
            ),
            HumanMessage(
                content=f"""
BLOG TOPIC:

{topic}


BLOG PLAN:

{plan}


RESEARCH:

{research}


CURRENT SECTION:

{task.title}


SECTION GOAL:

{task.goal}


KEY POINTS:

{bullets}
"""
            )
        ]
    )

    return {
        "sections": [
            {
                "id": task.id,
                "content": result.content
            }
        ]
    }

In [ ]:
def image_search(payload: dict) -> dict:

    task = payload["task"]

    query = task.image_query

    try:

        results = tavily.search(
            query=query,
            search_depth="basic",
            max_results=5,
            include_images=True
        )

        images = results.get("images", [])

        selected_images = []

        for image in images[:3]:

            selected_images.append(
                {
                    "task_id": task.id,
                    "query": query,
                    "url": image
                }
            )

        return {
            "images": selected_images
        }

    except Exception as e:

        print(
            f"Image search failed for '{query}': {e}"
        )

        return {
            "images": []
        }


In [ ]:
def image_fanout(state: State):

    return [
        Send(
            "image_search",
            {
                "task": task
            }
        )
        for task in state["plan"].tasks
    ]

In [ ]:
def reducer(state: State) -> dict:

    plan = state["plan"]

    sections = state.get("sections", [])

    images = state.get("images", [])

    # Sort sections by task ID
    sections = sorted(
        sections,
        key=lambda x: x["id"]
    )

    # Sort images by task ID
    images = sorted(
        images,
        key=lambda x: x["task_id"]
    )

    final_sections = []

    for section in sections:

        section_id = section["id"]

        content = section["content"]

        # Find image for this section
        section_images = [
            image
            for image in images
            if image["task_id"] == section_id
        ]

        if section_images:

            image = section_images[0]

            content += (
                f'\n\n'
                f'![Relevant image]({image["url"]})'
            )

        final_sections.append(content)

    body = "\n\n".join(final_sections)

    final_md = (
        f"# {plan.blog_title}\n\n"
        f"{body}"
    )

    return {
        "final": final_md
    }

In [ ]:
def editor(state: State) -> dict:

    result = llm.invoke(
        [
            SystemMessage(
                content="""
You are a senior technical editor.

Review the complete article.

Improve:

- Logical flow
- Grammar
- Technical clarity
- Section transitions
- Repetition
- Markdown formatting
- Consistency
- Readability
- Accuracy

Preserve the useful technical content.

Do not remove image Markdown.

Do not add unrelated information.

Do not write commentary about your editing process.

Return ONLY the final polished Markdown article.
"""
            ),
            HumanMessage(
                content=f"""
TOPIC:

{state["topic"]}


TARGET AUDIENCE:

{state["plan"].audience}


ARTICLE:

{state["final"]}
"""
            )
        ]
    )

    return {
        "final": result.content
    }


In [ ]:
graph = StateGraph(State)


graph.add_node(
    "orchestrator",
    orchestrator
)

graph.add_node(
    "researcher",
    researcher
)

graph.add_node(
    "worker",
    worker
)

graph.add_node(
    "image_search",
    image_search
)

graph.add_node(
    "reducer",
    reducer
)

graph.add_node(
    "editor",
    editor
)

In [ ]:
graph.add_edge(
    START,
    "orchestrator"
)

graph.add_edge(
    "orchestrator",
    "researcher"
)

# Research → parallel writers
graph.add_conditional_edges(
    "researcher",
    fanout,
    ["worker"]
)

# Research → parallel image searches
graph.add_conditional_edges(
    "researcher",
    image_fanout,
    ["image_search"]
)

# Workers → reducer
graph.add_edge(
    "worker",
    "reducer"
)

# Image search → reducer
graph.add_edge(
    "image_search",
    "reducer"
)

# Reducer → editor
graph.add_edge(
    "reducer",
    "editor"
)

# Editor → END
graph.add_edge(
    "editor",
    END
)

In [ ]:
app = graph.compile()

In [ ]:
result = app.invoke(
    {
        "topic": "How RAG works with LangGraph",
        "sections": [],
        "images": []
    },
    {
        "configurable": {
            "max_concurrency": 5
        }
    }
)

print(result["final"])